<a href="https://colab.research.google.com/github/SaloneJJ/FHT-Structures-ans-Sperner-Enumerations/blob/main/FHT_SaloneJJ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import itertools
time_import = __import__('time')
from collections import defaultdict
import networkx as nx

# ==========================================
# 1. VALIDATEUR ET FORMES CANONIQUES
# ==========================================

def get_flat_vertices(item):
    if isinstance(item, int):
        return {item}
    verts = set()
    for sub in item:
        verts.update(get_flat_vertices(sub))
    return verts

def is_connected_flat_hypergraph(hg, vertices):
    used = set()
    for e in hg:
        used.update(get_flat_vertices(e))
    if used != set(vertices):
        return False

    edges = list(hg)
    m = len(edges)
    if m <= 1:
        return True

    adj = {i: [] for i in range(m)}
    for i in range(m):
        for j in range(i+1, m):
            if get_flat_vertices(edges[i]).intersection(get_flat_vertices(edges[j])):
                adj[i].append(j)
                adj[j].append(i)

    visited = {0}
    queue = [0]
    while queue:
        curr = queue.pop(0)
        for neighbor in adj[curr]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return len(visited) == m

def validate_flat_fht(hg_set, vertices):
    elements = list(hg_set)
    if len(elements) == 0:
        return False
    flat_sets = [get_flat_vertices(e) for e in elements]
    for i in range(len(flat_sets)):
        for j in range(len(flat_sets)):
            if i != j and flat_sets[i].issubset(flat_sets[j]):
                return False
    return is_connected_flat_hypergraph(hg_set, vertices)

def apply_mapping(item, mapping):
    if isinstance(item, int):
        return mapping.get(item, item)
    return frozenset(apply_mapping(sub, mapping) for sub in item)

def relabel_item(item, mapping):
    if isinstance(item, int):
        return mapping[item]
    return frozenset(relabel_item(sub, mapping) for sub in item)

def get_standardized_form(hg):
    atoms = sorted(list(get_flat_vertices(hg)))
    mapping = {atom: i for i, atom in enumerate(atoms)}
    return relabel_item(hg, mapping)

def canon_tuple(item):
    if isinstance(item, int):
        return (0, item)
    sorted_subs = sorted([canon_tuple(sub) for sub in item])
    return (1, tuple(sorted_subs))

def get_integer_partitions(n):
    def partition(n, max_val, prefix):
        if n == 0:
            yield prefix
            return
        for i in range(min(max_val, n), 0, -1):
            yield from partition(n - i, i, prefix + [i])
    return list(partition(n, n, []))

def generate_sperner_antichains(edges):
    def backtrack(start_idx, current_chain):
        yield current_chain
        for i in range(start_idx, len(edges)):
            candidate = edges[i]
            if all(not candidate.issubset(e) and not e.issubset(candidate) for e in current_chain):
                current_chain.append(candidate)
                yield from backtrack(i + 1, current_chain)
                current_chain.pop()
    for chain in backtrack(0, []):
        if chain:
            yield frozenset(chain)

FHT_CACHE = {}

def get_all_fht_structures(n):
    if n in FHT_CACHE:
        return FHT_CACHE[n]

    vertices = list(range(1, n+1))
    structures = set()
    perms = list(itertools.permutations(vertices))

    # --- PHASE 1 : Structures plates (S(n)) ---
    all_flat_edges = [frozenset(comb) for k in range(2, n+1) for comb in itertools.combinations(vertices, k)]

    for hg_set in generate_sperner_antichains(all_flat_edges):
        if validate_flat_fht(hg_set, vertices):
            min_canon = min(
                canon_tuple(get_standardized_form(
                    frozenset(apply_mapping(e, {vertices[i]: p[i] for i in range(n)}) for e in hg_set)
                )) for p in perms
            )
            structures.add(min_canon)

    # --- PHASE 2 : Structures hiérarchiques ---
    if n >= 3:
        partitions = get_integer_partitions(n)
        for p in partitions:
            if len(p) > 1:
                unique_sizes = set(p)
                sub_fhts_map = {}
                for s in unique_sizes:
                    if s == 1:
                        sub_fhts_map[1] = [1]
                    elif s == 2:
                        sub_fhts_map[2] = [frozenset({1, 2})]
                    else:
                        valid_sub = []
                        for sf in get_all_fht_structures(s):
                            raw = untag_static(sf)
                            if len(raw) == 1 and len(get_flat_vertices(list(raw)[0])) == s:
                                continue
                            valid_sub.append(sf)
                        sub_fhts_map[s] = valid_sub

                def generate_hierarchical_for_partition(verts, remaining_parts):
                    if not remaining_parts:
                        yield []
                        return
                    current_size = remaining_parts[0]
                    rest_parts = remaining_parts[1:]

                    for comb in itertools.combinations(verts, current_size):
                        leftover = [v for v in verts if v not in comb]
                        for sf in sub_fhts_map[current_size]:
                            for rest_assignment in generate_hierarchical_for_partition(leftover, rest_parts):
                                yield [(list(comb), sf)] + rest_assignment

                for assignment in generate_hierarchical_for_partition(vertices, p):
                    block_choice = []
                    for block, sf in assignment:
                        size = len(block)
                        if size == 1:
                            block_choice.append(block[0])
                        elif size == 2:
                            block_choice.append(frozenset(block))
                        else:
                            atoms = sorted(list(get_flat_vertices(sf)))
                            mapping = {atoms[j]: block[j] for j in range(size)}
                            block_choice.append(apply_mapping(sf, mapping))

                    hierarchical_hg = frozenset([frozenset(block_choice)])
                    min_canon = min(
                        canon_tuple(get_standardized_form(
                            frozenset(apply_mapping(item, {vertices[i]: p_val[i] for i in range(n)}) for item in hierarchical_hg)
                        )) for p_val in perms
                    )
                    structures.add(min_canon)

    final_structures = []
    for c in structures:
        raw_hg = untag_static(c)
        atoms = sorted(list(get_flat_vertices(raw_hg)))
        mapping = {atoms[i]: i+1 for i in range(len(atoms))}
        final_structures.append(relabel_item(raw_hg, mapping))

    FHT_CACHE[n] = final_structures
    return FHT_CACHE[n]

def untag_static(t):
    if isinstance(t, int):
        return t
    if isinstance(t, tuple) and len(t) == 2:
        tag, val = t
        if tag == 0:
            return val
        return frozenset(untag_static(sub) for sub in val)
    return t


# ==========================================
# 2. ANALYSE ET EXECUTION
# ==========================================

def format_set(item):
    if isinstance(item, int):
        return str(item)
    sorted_items = sorted(list(item), key=lambda x: str(x))
    return "{" + ", ".join(format_set(e) for e in sorted_items) + "}"

def build_fht_dag(hg, n):
    G = nx.DiGraph()
    for i in range(1, n+1):
        G.add_node(i, type='atom')
    node_counter = n + 1

    def add_set(s):
        nonlocal node_counter
        current_node = node_counter
        G.add_node(current_node, type='set', size=len(s))
        node_counter += 1
        for item in s:
            if isinstance(item, int):
                G.add_edge(current_node, item)
            else:
                child_node = add_set(item)
                G.add_edge(current_node, child_node)
        return current_node

    add_set(hg)
    return G

def analyze_fht_networkx(hg, n):
    G = build_fht_dag(hg, n)
    def node_match(n1, n2):
        return n1['type'] == n2['type'] and n1.get('size') == n2.get('size')

    GM = nx.algorithms.isomorphism.DiGraphMatcher(G, G, node_match=node_match)
    automorphisms = []
    for iso in GM.isomorphisms_iter():
        mapping = {v: iso[v] for v in range(1, n+1)}
        if mapping not in automorphisms:
            automorphisms.append(mapping)

    card_aut = len(automorphisms)
    card_inv = sum(1 for v in range(1, n+1) if all(m[v] == v for m in automorphisms))

    parent = {v: v for v in range(1, n+1)}
    def find(i):
        if parent[i] == i: return i
        parent[i] = find(parent[i])
        return parent[i]
    def union(i, j):
        root_i, root_j = find(i), find(j)
        if root_i != root_j: parent[root_i] = root_j

    for mapping in automorphisms:
        for v in range(1, n+1): union(v, mapping[v])

    num_orbits = len({find(v) for v in range(1, n+1)})
    return card_aut, card_inv, num_orbits

if __name__ == "__main__":
    n = int(input("Entrez la valeur de n (nombre de sommets, ex: 3, 4, 5) : "))

    print(f"\nGénération des structures FHT pour n = {n}...")
    start_time = time_import.time()

    fht_structures = get_all_fht_structures(n)

    elapsed = time_import.time() - start_time
    print(f"Total de structures FHT validées : {len(fht_structures)} (Calculé en {elapsed:.2f} secondes)\n")

    print("Analyse des automorphismes via NetworkX en cours...")
    analyzed_data = []
    for idx, hg in enumerate(fht_structures, 1):
        card_aut, card_inv, num_orbits = analyze_fht_networkx(hg, n)
        analyzed_data.append({
            'id': idx,
            'representation': format_set(hg),
            'card_aut': card_aut,
            'card_inv': card_inv,
            'num_orbits': num_orbits
        })

    analyzed_data.sort(key=lambda x: (x['card_aut'], x['card_inv'], x['num_orbits']))

    grouped_data = defaultdict(list)
    for item in analyzed_data:
        key = (item['card_aut'], item['card_inv'], item['num_orbits'])
        grouped_data[key].append(item)

    print("=" * 70)
    print(f" CLASSIFICATION DES STRUCTURES FHT (n = {n})")
    print("=" * 70)

    for (c_aut, c_inv, n_orb), items in grouped_data.items():
        print(f"\n[Propriétés] |Aut(H)| = {c_aut}  |  |Inv(H)| = {c_inv}  |  Orbites (k) = {n_orb}")
        print(f"Nombre de structures : {len(items)}")
        print("-" * 70)
        for item in items:
            print(f"  * FHT #{item['id']} -> H = {item['representation']}")
    print("=" * 70)

KeyboardInterrupt: Interrupted by user

In [44]:
import itertools
import time

def get_flat_vertices(item):
    if isinstance(item, int):
        return {item}
    verts = set()
    for sub in item:
        verts.update(get_flat_vertices(sub))
    return verts

def is_connected_flat_hypergraph(hg, vertices):
    used = set()
    for e in hg:
        used.update(get_flat_vertices(e))
    if used != set(vertices):
        return False
    edges = list(hg)
    m = len(edges)
    if m <= 1:
        return True
    adj = {i: [] for i in range(m)}
    for i in range(m):
        for j in range(i+1, m):
            if get_flat_vertices(edges[i]).intersection(get_flat_vertices(edges[j])):
                adj[i].append(j)
                adj[j].append(i)
    visited = {0}
    queue = [0]
    while queue:
        curr = queue.pop(0)
        for neighbor in adj[curr]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return len(visited) == m

def validate_flat_fht(hg_set, vertices):
    elements = list(hg_set)
    if len(elements) == 0:
        return False
    flat_sets = [get_flat_vertices(e) for e in elements]
    for i in range(len(flat_sets)):
        for j in range(len(flat_sets)):
            if i != j and flat_sets[i].issubset(flat_sets[j]):
                return False
    return is_connected_flat_hypergraph(hg_set, vertices)

def apply_mapping(item, mapping):
    if isinstance(item, int):
        return mapping.get(item, item)
    return frozenset(apply_mapping(sub, mapping) for sub in item)

def get_standardized_form(hg):
    atoms = sorted(list(get_flat_vertices(hg)))
    mapping = {atom: i for i, atom in enumerate(atoms)}
    return apply_mapping(hg, mapping)

def canon_tuple(item):
    if isinstance(item, int):
        return (0, item)
    sorted_subs = sorted([canon_tuple(sub) for sub in item])
    return (1, tuple(sorted_subs))

def get_sperner_count(n):
    vertices = list(range(1, n+1))
    structures = set()
    perms = list(itertools.permutations(vertices))
    all_flat_edges = [frozenset(comb) for k in range(2, n+1) for comb in itertools.combinations(vertices, k)]

    def generate_sperner_antichains(edges):
        def backtrack(start_idx, current_chain):
            yield current_chain
            for i in range(start_idx, len(edges)):
                candidate = edges[i]
                if all(not candidate.issubset(e) and not e.issubset(candidate) for e in current_chain):
                    current_chain.append(candidate)
                    yield from backtrack(i + 1, current_chain)
                    current_chain.pop()
        for chain in backtrack(0, []):
            if chain:
                yield frozenset(chain)

    for hg_set in generate_sperner_antichains(all_flat_edges):
        if validate_flat_fht(hg_set, vertices):
            min_canon = min(
                canon_tuple(get_standardized_form(
                    frozenset(apply_mapping(e, {vertices[i]: p[i] for i in range(n)}) for e in hg_set)
                )) for p in perms
            )
            structures.add(min_canon)
    return len(structures)

if __name__ == "__main__":
    n = int(input("Entrez la valeur de n pour les hypergraphes de Sperner (ex: 3, 4, 5) : "))

    print(f"\nCalcul en cours pour n = {n}...")
    start_time = time.time()

    s_val = get_sperner_count(n)

    elapsed = time.time() - start_time
    print(f"S({n}) = {s_val} (Calculé en {elapsed:.2f} secondes)\n")

Entrez la valeur de n pour les hypergraphes de Sperner (ex: 3, 4, 5) : 5

Calcul en cours pour n = 5...
S(5) = 157 (Calculé en 25.44 secondes)

